In [65]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [66]:
### core fundataion - llm
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

### Node 1: JD Parser

In [67]:
from pydantic import BaseModel, Field
from typing import List

class JDStructured(BaseModel):
    role: str = Field(description="Job role/title")
    required_skills: List[str] = Field(description="Must-have skills")
    experience_years: int = Field(description="Years of experience required")
    location: str = Field(description="Job location")

In [68]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert recruiter who extracts structured data from job descriptions."),
    ("user", """
Extract the following from the job description:

- role
- required_skills
- experience_years
- location

Return ONLY valid JSON.

Job Description:
{jd}
""")
])

In [69]:
structured_llm = llm.with_structured_output(JDStructured)

chain = prompt | structured_llm

In [70]:
jd_text = """
We are looking for a Data Scientist with 3+ years of experience.
Must have strong Python, Machine Learning, and SQL skills.
Experience with AWS is a plus.
Location: Bangalore
"""

jd_parsed = chain.invoke({"jd": jd_text})

print(jd_parsed)

role='Data Scientist' required_skills=['Python', 'Machine Learning', 'SQL'] experience_years=3 location='Bangalore'


In [71]:
jd_parsed

JDStructured(role='Data Scientist', required_skills=['Python', 'Machine Learning', 'SQL'], experience_years=3, location='Bangalore')

### Node 2: Candidate Retriever

In [72]:
### create the candidate profile

candidates = [
    {
        "id": 1,
        "name": "Rahul Sharma",
        "skills": ["ML","Python","SQL"],
        "experience": 3,
        "location": "Bangalore",
        "summary": "Data Scientist with ML and analytics experience"
    },
    {
        "id": 2,
        "name": "Ankit Verma",
        "skills": ["Java", "Spring", "AWS"],
        "experience": 5,
        "location": "Delhi",
        "summary": "Backend engineer with cloud expertise"
    },
    {
        "id": 3,
        "name": "Priya Singh",
        "skills": ["Python", "Deep Learning", "NLP"],
        "experience": 4,
        "location": "Bangalore",
        "summary": "AI Engineer specializing in NLP"
    }
]

In [76]:
import json
from functools import lru_cache
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ✅ FIXED SCHEMA (List-based, NOT dict)
class SkillItem(BaseModel):
    skill: str
    synonyms: List[str]


class SkillList(BaseModel):
    skills: List[SkillItem]


# Prompt
skill_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a recruiting expert. Return ONLY valid JSON."),
    ("user", """
Expand each skill into synonyms, abbreviations, and closely related terms.

Role: {role}
Skills: {skills}

Rules:
- Include original skill
- Include abbreviations (ML, NLP, etc.)
- Include tools/frameworks
- Max 5–7 per skill

Return JSON like:
{{
  "skills": [
    {{"skill": "python", "synonyms": ["python", "py"]}},
    {{"skill": "machine learning", "synonyms": ["ml", "machine learning"]}}
  ]
}}
""")
])


# Chain
skill_chain = skill_prompt | llm.with_structured_output(SkillList)


# Cached function
@lru_cache(maxsize=32)
def expand_skills_with_llm(required_skills_tuple: tuple, role: str = ""):
    required_skills = list(required_skills_tuple)

    result = skill_chain.invoke({
        "role": role,
        "skills": json.dumps(required_skills)
    })

    # ✅ Convert list → dict (your expected format)
    skill_map = {}

    for item in result.skills:
        key = item.skill.lower().strip()
        synonyms = list(set([s.lower().strip() for s in item.synonyms]))
        skill_map[key] = synonyms

    return skill_map

In [77]:
import re
import json
from typing import List, Dict

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from sklearn.metrics.pairwise import cosine_similarity

#from nodes.skill_expander import expand_skills_with_llm


# LLM + Embeddings
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# -------------------------------
# Helpers
# -------------------------------
def _normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().strip())


# -------------------------------
# 1. KEYWORD FILTER
# -------------------------------
def keyword_prefilter(
    candidates: List[Dict],
    required_skills: List[str],
    skill_synonyms: Dict[str, List[str]],
    min_experience: int = 0,
    skill_match_threshold: float = 0.5,
) -> List[Dict]:

    shortlisted = []

    for candidate in candidates:

        if candidate.get("experience", 0) < min_experience:
            continue

        candidate_text = _normalize(
            " ".join(candidate.get("skills", []))
            + " " + candidate.get("summary", "")
        )

        matched_skills = []

        for req_skill in required_skills:
            skill_key = req_skill.lower().strip()
            synonyms = skill_synonyms.get(skill_key, [skill_key])

            pattern = r"\b(" + "|".join(re.escape(s) for s in synonyms) + r")\b"

            if re.search(pattern, candidate_text):
                matched_skills.append(req_skill)

        match_ratio = len(matched_skills) / len(required_skills) if required_skills else 0

        if match_ratio >= skill_match_threshold:
            shortlisted.append({
                **candidate,
                "_keyword_matched_skills": matched_skills,
                "_keyword_match_ratio": round(match_ratio, 2),
            })

    return shortlisted


# -------------------------------
# 2. EMBEDDING RERANK
# -------------------------------
def embedding_rerank(candidates, jd_parsed, top_n=10):

    jd_text = f"""
    Role: {jd_parsed.get('role')}
    Skills: {', '.join(jd_parsed.get('required_skills', []))}
    Experience: {jd_parsed.get('experience_years')}
    """

    jd_embedding = embeddings.embed_query(jd_text)

    for candidate in candidates:
        cand_text = f"""
        Skills: {', '.join(candidate.get('skills', []))}
        Experience: {candidate.get('experience')}
        Summary: {candidate.get('summary')}
        """

        cand_embedding = embeddings.embed_query(cand_text)

        score = cosine_similarity(
            [jd_embedding],
            [cand_embedding]
        )[0][0]

        candidate["_embedding_score"] = round(score * 100, 1)

    return sorted(candidates, key=lambda x: x["_embedding_score"], reverse=True)[:top_n]


# -------------------------------
# 3. LLM SCORING
# -------------------------------
class CandidateScore(BaseModel):
    match_score: int
    matched_skills: List[str]
    missing_skills: List[str]
    explanation: str


scoring_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a strict recruiter. Score candidate vs JD."),
    ("user", """
JD:
{jd}

Candidate:
{candidate}

Return:
- match_score (0-100)
- matched_skills
- missing_skills
- explanation (2 lines)
""")
])


scoring_chain = scoring_prompt | llm.with_structured_output(CandidateScore)


def llm_score(candidates, jd_parsed):

    scored = []

    for candidate in candidates:

        result = scoring_chain.invoke({
            "jd": json.dumps(jd_parsed),
            "candidate": json.dumps(candidate)
        })

        scored.append({
            **candidate,
            "match_score": result.match_score,
            "matched_skills": result.matched_skills,
            "missing_skills": result.missing_skills,
            "match_explanation": result.explanation,
        })

    return sorted(scored, key=lambda x: x["match_score"], reverse=True)


# -------------------------------
# 4. PIPELINE (Jupyter)
# -------------------------------
def run_candidate_pipeline(jd_parsed, candidates):

    # FIX: convert Pydantic → dict
    if not isinstance(jd_parsed, dict):
        jd_parsed = jd_parsed.model_dump()

    print(f"[filter] Total candidates: {len(candidates)}")

    # Step 1 — Skill Expansion
    skill_synonyms = expand_skills_with_llm(
        tuple(jd_parsed.get("required_skills", [])),
        role=jd_parsed.get("role", "")
    )

    print("[filter] Skill Synonyms:", json.dumps(skill_synonyms, indent=2))

    # Step 2 — Keyword filter
    prefiltered = keyword_prefilter(
        candidates=candidates,
        required_skills=jd_parsed.get("required_skills", []),
        skill_synonyms=skill_synonyms,
        min_experience=jd_parsed.get("experience_years", 0) - 1,
        skill_match_threshold=0.5,
    )

    print(f"[filter] After keyword filter: {len(prefiltered)}")

    # Step 3 — Embedding rerank (conditional)
    if len(prefiltered) > 5:
        reranked = embedding_rerank(prefiltered, jd_parsed)
    else:
        reranked = prefiltered

    print(f"[filter] After rerank: {len(reranked)}")

    # Step 4 — LLM scoring
    scored = llm_score(reranked, jd_parsed)

    print(f"[filter] Final candidates: {len(scored)}")

    return {
        "scored_candidates": scored,
        "skill_synonyms": skill_synonyms
    }

In [78]:
result = run_candidate_pipeline(jd_parsed, candidates)

for c in result["scored_candidates"]:
    print(c["name"], c["match_score"])

[filter] Total candidates: 3
[filter] Skill Synonyms: {
  "python": [
    "py",
    "anaconda",
    "python",
    "python3",
    "jupyter"
  ],
  "machine learning": [
    "deep learning",
    "supervised learning",
    "machine learning",
    "ml",
    "unsupervised learning"
  ],
  "sql": [
    "pl/sql",
    "sql",
    "structured query language",
    "t-sql",
    "mysql"
  ]
}
[filter] After keyword filter: 2
[filter] After rerank: 2
[filter] Final candidates: 2
Rahul Sharma 100
Priya Singh 70


### Node 4: Outreach Agent

In [79]:
from pydantic import BaseModel

class CandidateReply(BaseModel):
    reply: str

class InterestScore(BaseModel):
    interest_score: int
    explanation: str

In [80]:
from langchain_core.prompts import ChatPromptTemplate

reply_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a job candidate responding to a recruiter.

Be realistic:
- Some candidates are actively looking
- Some are passive
- Some are not interested

Keep response natural and human-like.
"""),

    ("user", """
Candidate Profile:
{candidate}

Recruiter Message:
{message}

Reply as the candidate.
""")
])

In [81]:
score_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an expert recruiter.

Based on the conversation, estimate candidate interest.

Return:
- interest_score (0-100)
- explanation (1-2 lines)

Guidelines:
0-30 → Not interested  
30-60 → Passive  
60-100 → Interested  
"""),

    ("user", """
Conversation:
{conversation}
""")
])

In [82]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

reply_chain = reply_prompt | llm.with_structured_output(CandidateReply)
score_chain = score_prompt | llm.with_structured_output(InterestScore)

In [98]:
import random

def run_outreach(state: dict) -> dict:

    jd = state["jd_parsed"]
    if not isinstance(jd, dict):
        jd = jd.model_dump()

    for sc in state["scored_candidates"]:

        # Skip low matches
        if sc["match_score"] < 40:
            sc["interest_score"] = 0
            sc["interest_explanation"] = "Low match score"
            sc["conversation_log"] = []
            sc["response_status"] = "Not Contacted"
            continue

        candidate = sc

        # -------------------------------
        # Turn 1 — Agent Message
        # -------------------------------
        opening = f"""
Hi {candidate.get('name')},

We have a {jd.get('seniority', 'mid-level')} role in {jd.get('domain', 'data science')}.

Your experience in {', '.join(candidate.get('skills', [])[:2])} looks like a strong fit.

Would you be open to exploring this opportunity?
"""

        # -------------------------------
        # NEW: Simulate No Response
        # -------------------------------
        no_response_prob = 0.2   # 20% ghost rate

        if random.random() < no_response_prob:
            sc["interest_score"] = 0
            sc["interest_explanation"] = "Candidate did not respond"
            sc["conversation_log"] = [
                {"role": "agent", "content": opening}
            ]
            sc["response_status"] = "No Response"

            sc["final_score"] = round(
                0.7 * sc["match_score"], 2
            )
            continue

        # -------------------------------
        # Turn 2 — Candidate Reply
        # -------------------------------
        reply_obj = reply_chain.invoke({
            "candidate": json.dumps(candidate),
            "message": opening
        })

        reply = reply_obj.reply

        conversation = [
            {"role": "agent", "content": opening},
            {"role": "candidate", "content": reply}
        ]

        # -------------------------------
        # Turn 3 — Interest Score
        # -------------------------------
        score_obj = score_chain.invoke({
            "conversation": json.dumps(conversation)
        })

        sc["interest_score"] = score_obj.interest_score
        sc["interest_explanation"] = score_obj.explanation
        sc["conversation_log"] = conversation
        sc["response_status"] = "Responded"

        sc["final_score"] = round(
            0.7 * sc["match_score"] + 0.3 * sc["interest_score"], 2
        )

    # Sort
    state["scored_candidates"] = sorted(
        state["scored_candidates"],
        key=lambda x: x.get("final_score", 0),
        reverse=True
    )

    return state

In [100]:
state = {
    "jd_parsed": jd_parsed,
    "scored_candidates": result["scored_candidates"]
}

state = run_outreach(state)

for c in state["scored_candidates"]:
    print(c["name"])
    print("Match:", c["match_score"])
    print("Interest:", c["interest_score"])
    print("Final:", c["final_score"])
    print("Status:", c.get("response_status"))

    reply = (
        c["conversation_log"][1]["content"]
        if len(c.get("conversation_log", [])) > 1
        else "No response"
    )

    print("Reply:", reply)
    print("-" * 50)

Priya Singh
Match: 70
Interest: 85
Final: 74.5
Status: Responded
Reply: Hi there,

Thank you for considering me for the data science role! I’m really excited about the potential to contribute my skills in Python and Deep Learning to your team. 

I understand that SQL is also a requirement, and while I don’t have that experience yet, I'm keen on learning and can quickly pick up the skill if given the opportunity. 

I’d love to discuss this role further and see how I might be able to fit into your team.

Looking forward to hearing from you!

Best regards,
Priya
--------------------------------------------------
Rahul Sharma
Match: 100
Interest: 0
Final: 70.0
Status: No Response
Reply: No response
--------------------------------------------------


### Node : Reranker

In [101]:
from pydantic import BaseModel

class ConversationSummary(BaseModel):
    summary: str

In [102]:
from langchain_core.prompts import ChatPromptTemplate

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You summarize recruiter-candidate conversations."),
    ("user", """
Conversation:
{conversation}

Summarize in 1-2 lines focusing on:
- candidate interest
- any concerns
""")
])

In [103]:
summary_chain = summary_prompt | llm.with_structured_output(ConversationSummary)

In [104]:
def assign_status(final_score):
    if final_score >= 85:
        return "High Priority"
    elif final_score >= 65:
        return "Consider"
    else:
        return "Low Priority"

In [107]:
def rerank_and_format(state: dict) -> dict:

    candidates = state["scored_candidates"]
    jd = state["jd_parsed"]

    if not isinstance(jd, dict):
        jd = jd.model_dump()

    final_output = []

    for c in candidates:

        # -------------------------------
        # Scores
        # -------------------------------
        match_score = c.get("match_score", 0)
        interest_score = c.get("interest_score", 0)

        final_score = round(
            0.6 * match_score + 0.4 * interest_score,
            2
        )

        # -------------------------------
        # Conversation Summary
        # -------------------------------
        conversation = c.get("conversation_log", [])

        if conversation:
            summary_obj = summary_chain.invoke({
                "conversation": str(conversation)
            })
            summary = summary_obj.summary
        else:
            summary = "No interaction"

        # -------------------------------
        # Explanation
        # -------------------------------
        explanation = f"""
Matched Skills: {', '.join(c.get('matched_skills', []))}
Missing Skills: {', '.join(c.get('missing_skills', []))}
Candidate Interest Score: {interest_score}
"""

        # -------------------------------
        # Final Object (UPDATED)
        # -------------------------------
        final_output.append({
            "candidate_name": c.get("name"),
            "match_score": match_score,
            "interest_score": interest_score,
            "final_score": final_score,
            "status": assign_status(final_score),
            "explanation": explanation.strip(),
            "conversation_summary": summary
        })

    # Sort
    final_output = sorted(
        final_output,
        key=lambda x: x["final_score"],
        reverse=True
    )

    state["final_output"] = final_output
    return state

In [108]:
state = run_outreach(state)
state = rerank_and_format(state)

for c in state["final_output"]:
    print("⭐", c["candidate_name"])
    print("Match Score:", c["match_score"])
    print("Interest Score:", c["interest_score"])
    print("Final Score:", c["final_score"])
    print("Status:", c["status"])
    print("Explanation:", c["explanation"])
    print("Summary:", c["conversation_summary"])
    print("=" * 60)

⭐ Rahul Sharma
Match Score: 100
Interest Score: 20
Final Score: 68.0
Status: Consider
Explanation: Matched Skills: Python, Machine Learning, SQL
Missing Skills: 
Candidate Interest Score: 20
Summary: Rahul Sharma expressed gratitude for the opportunity in data science but indicated he is not actively looking for new roles at this time, as he is focused on his current work.
⭐ Priya Singh
Match Score: 70
Interest Score: 30
Final Score: 54.0
Status: Low Priority
Explanation: Matched Skills: Python, Machine Learning
Missing Skills: SQL
Candidate Interest Score: 30
Summary: Priya Singh expressed gratitude for the opportunity in data science but noted that she is not currently seeking new roles due to focus on her existing projects. She is open to staying in touch for future possibilities.


In [109]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [ ]:
def export_to_csv(final_output, filename="shortlisted_candidates.csv"):

    rows = []

    for c in final_output:
        rows.append({
            "Candidate Name": c.get("candidate_name"),
            "Match Score": c.get("match_score"),
            "Interest Score": c.get("interest_score"),
            "Final Score": c.get("final_score"),
            "Status": c.get("status"),
            "Explanation": c.get("explanation"),
            "Conversation Summary": c.get("conversation_summary")
        })

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)

    print(f"✅ CSV exported: {filename}")

In [ ]:
export_to_csv(state["final_output"])